In [6]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [7]:
api_key = os.environ["API_KEY"]
import dspy
lm = dspy.LM("deepseek/deepseek-v4-flash", temperature=1, api_key=api_key, max_tokens=64000,  api_base="https://api.deepseek.com")
dspy.configure(lm=lm)

In [9]:
from datasets import load_dataset

def init_dataset():
    train_split = load_dataset("AI-MO/aimo-validation-aime")['train']
    train_split = [
        dspy.Example({
            "problem": x['problem'],
            'solution': x['solution'],
            'answer': x['answer'],
        }).with_inputs("problem")
        for x in train_split
    ]
    import random
    random.Random(0).shuffle(train_split)
    tot_num = len(train_split)

    test_split = load_dataset("MathArena/aime_2025")['train']
    test_split = [
        dspy.Example({
            "problem": x['problem'],
            'answer': x['answer'],
        }).with_inputs("problem")
        for x in test_split
    ]

    train_set = train_split[:int(0.5 * tot_num)]
    val_set = train_split[int(0.5 * tot_num):]
    test_set = test_split * 5

    return train_set, val_set, test_set

In [10]:
train_set, val_set, test_set = init_dataset()

len(train_set), len(val_set), len(test_set)

README.md:   0%|          | 0.00/1.08k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/261k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/90 [00:00<?, ? examples/s]

README.md:   0%|          | 0.00/1.90k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/14.3k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/30 [00:00<?, ? examples/s]

(45, 45, 150)

In [11]:
print("Problem:")
print(train_set[0]['problem'])
print("\n\nSolution:")
print(train_set[0]['solution'])
print("\n\nAnswer:")
print(train_set[0]['answer'])

Problem:
In isosceles trapezoid $ABCD$, parallel bases $\overline{AB}$ and $\overline{CD}$ have lengths $500$ and $650$, respectively, and $AD=BC=333$. The angle bisectors of $\angle{A}$ and $\angle{D}$ meet at $P$, and the angle bisectors of $\angle{B}$ and $\angle{C}$ meet at $Q$. Find $PQ$.


Solution:
We have the following diagram:

Let $X$ and $W$ be the points where $AP$ and $BQ$ extend to meet $CD$, and $YZ$ be the height of $\triangle AZB$. As proven in Solution 2, triangles $APD$ and $DPW$ are congruent right triangles. Therefore, $AD = DW = 333$. We can apply this logic to triangles $BCQ$ and $XCQ$ as well, giving us $BC = CX = 333$. Since $CD = 650$, $XW = DW + CX - CD = 16$.
Additionally, we can see that $\triangle XZW$ is similar to $\triangle PQZ$ and $\triangle AZB$. We know that $\frac{XW}{AB} = \frac{16}{500}$. So, we can say that the height of the triangle $AZB$ is $500u$ while the height of the triangle $XZW$ is $16u$. After that, we can figure out the distance from 

In [12]:
class GenerateResponse(dspy.Signature):
    """Solve the problem and provide the answer in the correct format."""
    problem = dspy.InputField()
    answer = dspy.OutputField()

program = dspy.ChainOfThought(GenerateResponse)

In [13]:
def metric(example, prediction, trace=None, pred_name=None, pred_trace=None):
    correct_answer = int(example['answer'])
    try:
        llm_answer = int(prediction.answer)
    except ValueError as e:
        return 0
    return int(correct_answer == llm_answer)

In [14]:
import dspy
evaluate = dspy.Evaluate(
    devset=test_set,
    metric=metric,
    num_threads=32,
    display_table=True,
    display_progress=True
)

evaluate(program)

Average Metric: 123.00 / 125 (98.4%):  83%|█████████████████████████████████████████████████████████████████████████████████████████▏                 | 125/150 [03:46<02:31,  6.04s/it]

2026/05/30 19:51:28 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=32000. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 1)  if the reason for truncation is repetition.


Average Metric: 124.00 / 126 (98.4%):  84%|█████████████████████████████████████████████████████████████████████████████████████████▉                 | 126/150 [03:59<02:57,  7.39s/it]

2026/05/30 19:51:43 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=32000. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 1)  if the reason for truncation is repetition.
2026/05/30 19:51:46 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=32000. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 1)  if the reason for truncation is repetition.


Average Metric: 126.00 / 128 (98.4%):  85%|███████████████████████████████████████████████████████████████████████████████████████████▎               | 128/150 [04:13<02:38,  7.21s/it]

...

2026/05/30 19:57:18 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=32000. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 1)  if the reason for truncation is repetition.
2026/05/30 19:57:18 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=32000. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 1)  if the reason for truncation is repetition.
2026/05/30 19:57:18 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=32000. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy

Average Metric: 136.00 / 140 (97.1%):  99%|█████████████████████████████████████████████████████████████████████████████████████████████████████████▌ | 148/150 [09:37<00:07,  3.90s/it]

2026/05/30 19:57:18 ERROR dspy.utils.parallelizer: Error for Example({'problem': 'Let $N$ denote the numbers of ordered triples of positive integers $(a, b, c)$ such that $a, b, c \\le 3^6$ and $a^3 + b^3 + c^3$ is a multiple of $3^7$. Find the remainder when $N$ is divided by $1000$.', 'answer': 735}) (input_keys={'problem'}): The LM returned an empty or null response.

Adapter JSONAdapter failed to parse the LM response. 

LM Response: {'text': '', 'reasoning_content': "We need to count ordered triples (a,b,c) of positive integers ≤ 3^6 = 729 such that a^3+b^3+c^3 ≡ 0 mod 3^7. Since 3^7 = 2187. We are counting modulo 2187, but a,b,c up to 729. The condition involves cubes mod 3^7. We can use properties of cubes modulo powers of 3.\n\nWe need to count number of solutions in (ℤ/3^7ℤ)^3? But a,b,c are in range 1..729, which is exactly the residues modulo 729? Actually 3^6=729, so a,b,c are in [1,729]. Since 729 is not a multiple of 3^7, but note that modulo 3^7, numbers from 1 to 729 ar

Exception: Execution cancelled due to errors or interruption.

2026/05/30 19:57:51 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=32000. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 1)  if the reason for truncation is repetition.
2026/05/30 19:57:51 ERROR dspy.utils.parallelizer: Error for Example({'problem': 'Let $N$ denote the numbers of ordered triples of positive integers $(a, b, c)$ such that $a, b, c \\le 3^6$ and $a^3 + b^3 + c^3$ is a multiple of $3^7$. Find the remainder when $N$ is divided by $1000$.', 'answer': 735}) (input_keys={'problem'}): The LM returned an empty or null response.

Adapter JSONAdapter failed to parse the LM response. 

LM Response: {'text': '', 'reasoning_content': 'We are asked: "Let N denote the numbers of ordered triples of positive integers (a, b, c) such that a, b, c ≤ 3^6 and a^3 + b^3 + c^3 is a multiple of 3

In [15]:
def metric_with_feedback(example, prediction, trace=None, pred_name=None, pred_trace=None):
    correct_answer = int(example['answer'])
    written_solution = example.get('solution', '')
    try:
        llm_answer = int(prediction.answer)
    except ValueError as e:
        feedback_text = f"The final answer must be a valid integer and nothing else. You responded with '{prediction.answer}', which couldn't be parsed as a python integer. Please ensure your answer is a valid integer without any additional text or formatting."
        feedback_text += f" The correct answer is '{correct_answer}'."
        if written_solution:
            feedback_text += f" Here's the full step-by-step solution:\n{written_solution}\n\nThink about what takeaways you can learn from this solution to improve your future answers and approach to similar problems and ensure your final answer is a valid integer."
        return dspy.Prediction(score=0, feedback=feedback_text)

    score = int(correct_answer == llm_answer)

    feedback_text = ""
    if score == 1:
        feedback_text = f"Your answer is correct. The correct answer is '{correct_answer}'."
    else:
        feedback_text = f"Your answer is incorrect. The correct answer is '{correct_answer}'."
    
    if written_solution:
        feedback_text += f" Here's the full step-by-step solution:\n{written_solution}\n\nThink about what takeaways you can learn from this solution to improve your future answers and approach to similar problems."

    return dspy.Prediction(score=score, feedback=feedback_text)

In [16]:
from dspy import GEPA

optimizer = GEPA(
    metric=metric_with_feedback,
    auto="light",
    num_threads=32,
    track_stats=True,
    reflection_minibatch_size=3,
    reflection_lm=  dspy.LM("deepseek/deepseek-v4-flash", temperature=1, api_key=api_key, max_tokens=64000,  api_base="https://api.deepseek.com")
)

optimized_program = optimizer.compile(
    program,
    trainset=train_set,
    valset=val_set,
)

2026/05/30 20:00:32 INFO dspy.teleprompt.gepa.gepa: Running GEPA for approx 560 metric calls of the program. This amounts to 6.22 full evals on the train+val set.
2026/05/30 20:00:32 INFO dspy.teleprompt.gepa.gepa: Using 45 examples for tracking Pareto scores. You can consider using a smaller sample of the valset to allow GEPA to explore more diverse solutions within the same budget. GEPA requires you to provide the smallest valset that is just large enough to match your downstream task distribution, while providing as large trainset as possible.
GEPA Optimization:   0%|                                                                                                                                  | 0/560 [00:00<?, ?rollouts/s]2026/05/30 20:04:37 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=32000. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. 

Average Metric: 3.00 / 3 (100.0%): 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:50<00:00, 16.83s/it]

2026/05/30 20:12:16 INFO dspy.evaluate.evaluate: Average Metric: 3.0 / 3 (100.0%)
2026/05/30 20:12:16 INFO dspy.teleprompt.gepa.gepa: Iteration 1: All subsample scores perfect. Skipping.
2026/05/30 20:12:16 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Reflective mutation did not propose a new candidate
GEPA Optimization:   9%|██████████▏                                                                                                            | 48/560 [11:44<2:05:42, 14.73s/rollouts]2026/05/30 20:12:16 INFO dspy.teleprompt.gepa.gepa: Iteration 2: Selected program 0 score: 0.9555555555555556



Average Metric: 2.00 / 2 (100.0%):  67%|████████████████████████████████████████████████████████████████████████████                                      | 2/3 [01:15<00:41, 41.31s/it]

2026/05/30 20:16:04 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=32000. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 1)  if the reason for truncation is repetition.
2026/05/30 20:16:17 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=32000. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 1)  if the reason for truncation is repetition.


...

Average Metric: 3.00 / 3 (100.0%): 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [02:45<00:00, 55.14s/it]

2026/05/30 20:23:15 INFO dspy.evaluate.evaluate: Average Metric: 3.0 / 3 (100.0%)
2026/05/30 20:23:15 INFO dspy.teleprompt.gepa.gepa: Iteration 3: All subsample scores perfect. Skipping.
2026/05/30 20:23:15 INFO dspy.teleprompt.gepa.gepa: Iteration 3: Reflective mutation did not propose a new candidate
GEPA Optimization:  10%|████████████                                                                                                           | 57/560 [22:43<4:24:14, 31.52s/rollouts]2026/05/30 20:23:15 INFO dspy.teleprompt.gepa.gepa: Iteration 4: Selected program 0 score: 0.9555555555555556



Average Metric: 1.00 / 1 (100.0%):  33%|██████████████████████████████████████                                                                            | 1/3 [00:26<00:52, 26.49s/it]

2026/05/30 20:24:28 WARNING dspy.utils.parallelizer: SIGINT received. Cancelling.


KeyboardInterrupt: 

In [ ]:
print(optimized_program.predict.signature.instructions)

In [ ]:
evaluate(optimized_program)